# Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


# Setup

In [ ]:
!pip install -qU transformers==5.3.0 datasets==4.8.4 optimum==2.1.0
!pip install -qU wandb
!pip install -qU json-repair==0.58.6
!pip install -qU faker==40.11.1
!pip install -qU vllm==0.18.0
!pip install locust==2.43.3

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('hf')
!huggingface-cli login --token {hf_token}

# Imports

In [3]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests
import time

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

data_dir = "/gdrive/MyDrive/llm-finetuning/Jobs_Dataset"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

# Tasks

In [4]:
test_job_description = """
Headquarters: New York City, USA | Status: Full-Time

At GlobalData Corp, we are driving the future of enterprise decision-making. Established in 2005, we have been recognized as a leader in data analytics. We promote a culture of inclusion and offer fantastic benefits like unlimited PTO and premium health insurance. We are an Equal Opportunity Employer.

Currently, we are looking for a highly skilled Senior MLOps Engineer to join our thriving Data Science team in NYC. Note: this is a Hybrid role requiring 3 days a week in the office.

Core Duties:
Your typical day involves building and maintaining robust CI/CD pipelines for ML models using GitHub Actions and Jenkins. You will monitor production models for data drift with MLflow and Grafana. You'll also manage orchestration on our Kubernetes clusters and leverage Terraform to manage AWS infrastructure. Collaborating with Backend teams to deploy FastAPI microservices is a key part of the job.

Requirements:
We require a solid background in software engineering and cloud infrastructure. Specifically, we need 4+ years of hands-on MLOps experience in a production environment. Proficiency in Python and familiarity with tools like PyTorch, Docker, Snowflake, and ArgoCD is mandatory.

Compensation:
We offer a competitive base salary for this position, ranging from $150,000 to $180,000 annually, depending on performance.

Join GlobalData and help us shape the world of AI!
"""

# Details Extraction

In [5]:
ExperienceLevel = Literal["Junior", "Mid-level", "Senior", "Executive", "Not Specified"]
WorkModel = Literal["Remote", "On-site", "Hybrid", "Not Specified"]

class JobDetails(BaseModel):
    Job_Title: str = Field(..., description="The Arabic translation of the exact job title.")
    Company_Name: Optional[str] = Field(None, description="The company name, or null if strictly confidential or not mentioned.")
    Location: Optional[str] = Field(None, description="The Arabic translation of the location (e.g., 'القاهرة', 'عن بعد'), or null if not mentioned.")
    Experience_Level: ExperienceLevel = Field(..., description="Select strictly ONE option that matches the required experience level.")
    Min_Years_of_Experience: Optional[int] = Field(None, description="The MINIMUM number of years of experience required as an integer (e.g., 3), or null.")
    Max_Years_of_Experience: Optional[int] = Field(None, description="The MAXIMUM number of years of experience required as an integer (e.g., 5), or null.")
    Salary: Optional[str] = Field(None, description="The mentioned salary range or compensation translated to Arabic if needed, or null if not mentioned.")
    Tech_Stack: List[str] = Field(..., description="List of technical skills and tools ONLY, kept strictly in English. If none mentioned, return an empty list [].")
    Work_Model: WorkModel = Field(..., description="Select strictly ONE work model.")
    Core_Responsibilities: List[str] = Field(..., description="Extract 1-5 main responsibilities and translate them into professional Arabic.")


In [6]:

job_extraction_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are an expert HR Data Parser and a bilingual (English-Arabic) AI Engineer.",
            "Your task is to extract specific information from an English Job Description based on a provided Pydantic schema.",
            "CRITICAL RULES:",
            "1. JSON Keys MUST remain strictly in English.",
            "2. JSON Values MUST be translated into professional Arabic.",
            "3. Technical skills, frameworks, and tools (e.g., Python, AWS, React) MUST be kept in English.",
            "4. If a specific piece of information (like Company Name, Salary, or Years of Experience) is not mentioned, you MUST set its value to null.",
            "5. Do not generate any introduction, conclusion, or conversational text. Output ONLY a valid JSON object."
        ])
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Job Description:",
            test_job_description.strip(),
            "",
            "## Pydantic Schema:",
            json.dumps(
                JobDetails.model_json_schema(), ensure_ascii=False
            ),
            "",
            "## Extracted JSON:",
            "```json\n{"
        ])
    }
]

# Evaluation The Base Model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

In [ ]:
text = tokenizer.apply_chat_template(
    job_extraction_messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
print(response)

{
    "Job_Title": "مطور مهندسي معلومات متوسطة الخبرة",
    "Company_Name": null,
    "Location": "نيويورك، الولايات المتحدة",
    "Experience_Level": "متوسطة",
    "Min_Years_of_Experience": null,
    "Max_Years_of_Experience": null,
    "Salary": null,
    "Tech_Stack": [],
    "Work_Model": " hybrids",
    "Core_Responsibilities": [
        "بناء وصيانة الروتاريد CI/CD للعجلات التحليلية باستخدام GitHub Actions و Jenkins.",
        "مراقبة العجلات التحليلية في الوقت الحقيقي لـ data drift مع MLflow و Grafana.",
        "إدارة التجميع على كوكبنا Kubernetes وتقديم تطبيق Orchestration.",
        "تعاون مع فريق الخلفية لتنفيذ Microservices FastAPI."
    ]
}


# 📊 Pre-Training Model Evaluation (Baseline Zero-Shot Analysis)

**Evaluated Model:** `Qwen2.5-1.5B-Instruct`
**Evaluation Type:** Zero-Shot Prompting (No prior task-specific training)
**Objective:** Establish a performance baseline to measure improvements post-Supervised Fine-Tuning (SFT).

---

##  Executive Summary
The zero-shot baseline test reveals that while the model successfully follows basic structural instructions (producing valid JSON), it heavily fails at strict schema adherence, precise data extraction, and domain-specific translation. This outcome strongly validates the absolute necessity of conducting Supervised Fine-Tuning (SFT) using our custom domain-specific dataset.

---

## 🟢 1. Strengths & Successes (The Good)
Despite its limitations, the model responded well to advanced Prompt Engineering techniques:
* **✅ 100% Valid JSON Structure:** The model successfully generated a syntactically correct JSON object without any conversational filler or introductory text. This proves the effectiveness of the "pre-filling" technique (````json\n{`).
* **✅ Key Language Adherence:** It successfully kept all JSON keys in English, as strictly requested in the prompt.
* **✅ Partial Extraction Success:** Successfully extracted the location and translated it correctly (`"Location": "نيويورك، الولايات المتحدة"`).

---

## 🔴 2. Architectural Schema Violations (The Ugly)
The model completely broke the strict typing constraints required by the Pydantic backend schema:
* **❌ Ignored `Literal` Constraints:**
  * For `Experience_Level`: It translated the value to `"متوسطة"` instead of adhering to the mandatory English literal options (e.g., `"Mid-level"` or `"Senior"`).
  * For `Work_Model`: It generated `" hybrids"` (adding a leading space and a plural 's') instead of the exact required string `"Hybrid"`.
*(Note: In a real-world production environment, these violations would immediately crash a database expecting strict Enum values).*

---

## 🟡 3. Data Loss & Recall Issues (Cognitive Overload)
The complexity of the prompt (instructing the model to read English, extract data, format as JSON, and translate values to Arabic simultaneously) caused a cognitive overload for the 1.5B parameter model, leading to severe data loss:
* **❌ Tech Stack Blindness:** The model returned an empty array `[]` for `Tech_Stack`, completely failing to extract over 10 explicit technologies mentioned in the text (e.g., *MLflow, Kubernetes, FastAPI, Terraform*).
* **❌ Extraction Failure:** It failed to capture clearly stated entities and defaulted to `null` for:
  * Company Name (*GlobalData Corp*)
  * Salary (*$150,000 to $180,000*)
  * Minimum Years of Experience (*4*)

---

##  4. Translation Hallucinations
When forced to translate complex MLOps engineering terminology into Arabic, the small model resorted to guessing, resulting in comical and inaccurate translations:
1. **"العجلات التحليلية" (Analytical Wheels):** A bizarre mistranslation of the term `ML models`.
2. **"الروتاريد" (Al-Rotarid):** A completely made-up, non-existent Arabic word used to translate `Pipelines` (should be "مسارات").
3. **"على كوكبنا Kubernetes" (On our planet Kubernetes):** A hilarious mistranslation of `Clusters` (which means groups of servers, or "عناقيد"), where the model interpreted it literally as celestial clusters or planets.

---

###  Engineering Conclusion & Next Steps
This raw output represents our **Ground Zero Baseline**.
To resolve these critical failures, we will proceed with **Supervised Fine-Tuning (SFT)** using LLaMA Factory on our highly curated dataset of 400 records. The training objective is to reshape the model's attention mechanism so it learns to:
1. Strictly respect backend `Literal` types.
2. Aggressively hunt for and extract hidden technical stacks.
3. Utilize accurate, professional Arabic engineering terminology.

<hr>

# Format Finetuning Datasets for LlamaFactory



In [ ]:
input_file = os.path.join(data_dir, "jobs_dataset.json")
output_file = os.path.join(data_dir, "llama_factory_dataset.json")
system_prompt ="\n".join([
    "You are an expert bilingual AI data parser.",
    "Your task is to extract information from English text into a strictly valid JSON object, translating values to Arabic while keeping technical terms and JSON keys in English.",
    "CRITICAL RULE: Do not generate any introduction or conclusion."
    ])

try:
    with open(input_file, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    print(f"Successfully loaded {len(raw_data)} records.")
except FileNotFoundError:
    print(f"Error: Could not find the file at {input_file}. Please check the path and file name.")
    raw_data = []

# Process, Format, and Shuffle the data
if raw_data:
    formatted_data = []

    for item in raw_data:
        formatted_item = {
            "system": system_prompt,
            "instruction": item.get("instruction", ""),
            "input": item.get("input", ""),
            # Convert the nested JSON object into a formatted string wrapped in Markdown
            "output": "```json\n" + json.dumps(item.get("output", {}), ensure_ascii=False, indent=2) + "\n```"
        }
        formatted_data.append(formatted_item)

    # SHUFFLE THE DATA
    random.Random(101).shuffle(formatted_data)

    # Save the formatted dataset
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(formatted_data, f, ensure_ascii=False, indent=2)

    print("\n Success! The dataset is shuffled, formatted, and ready for LLaMA Factory.")

Successfully loaded 385 records.

 Success! The dataset is shuffled, formatted, and ready for LLaMA Factory.


In [ ]:
train_file = os.path.join(data_dir, "train.json")
val_file = os.path.join(data_dir, "val.json")

split_ratio = 0.90
split_index = int(len(formatted_data) * split_ratio)

train_data = formatted_data[:split_index]
val_data = formatted_data[split_index:]

print(f" Data split statistics:")
print(f" Training Set: {len(train_data)} records")
print(f" Validation Set: {len(val_data)} records")

with open(train_file, 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)

with open(val_file, 'w', encoding='utf-8') as f:
    json.dump(val_data, f, ensure_ascii=False, indent=2)

 Data split statistics:
 Training Set: 346 records
 Validation Set: 39 records


# Finetuning

In [ ]:
# Configure LLaMA-Factory for the new datasets

# update /content/LLaMA-Factory/data/dataset_info.json and append
# ```
   "jobs_train": {
        "file_name": "/gdrive/MyDrive/llm-finetuning/Jobs_Dataset/train.json"
        }

    "jobs_val": {
        "file_name": "/gdrive/MyDrive/llm-finetuning/Jobs_Dataset/val.json"
    }
# ```

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/test_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: jobs_train
eval_dataset: jobs_val
template: qwen
cutoff_len: 1024
max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: /content/test_jobs_model
logging_steps: 1
save_steps: 100
plot_loss: false
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 10

### Logging & Hub
report_to: none
push_to_hub: false

Writing /content/LLaMA-Factory/examples/train_lora/test_finetune.yaml


In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/test_finetune.yaml

2026-03-28 02:46:48.012874: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774666008.229551    8089 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774666008.290626    8089 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774666008.736598    8089 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774666008.736636    8089 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774666008.736640    8089 computation_placer.cc:177] computation placer alr

In [ ]:
%%writefile /content/LLaMA-Factory/examples/train_lora/jobs_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 32
lora_target: all

### dataset
dataset: jobs_train
eval_dataset: jobs_val
template: qwen
cutoff_len: 1024
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/llm-finetuning/models/checkpoint-200
output_dir: /gdrive/MyDrive/llm-finetuning/models/Jobs/
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: jobs-finetune-llamafactory

push_to_hub: true
export_hub_model_id: "abdoghazala7/Qwen-1.5B-Jobs-Parser"
hub_private_repo: false
hub_strategy: end

Writing /content/LLaMA-Factory/examples/train_lora/jobs_finetune.yaml


In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/jobs_finetune.yaml

# Finetuned Model Evaluation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

In [12]:
finetuned_model_id = "abdoghazala7/Jobs"
model.load_adapter(finetuned_model_id)

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/148M [00:00<?, ?B/s]

In [8]:
def genereate_response(job_description):

    text = tokenizer.apply_chat_template(
    job_description,
    tokenize=False,
    add_generation_prompt=True
                                        )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [ ]:
print(genereate_response(job_extraction_messages))

```json
{
    "Job_Title": "مهندس عمليات تعلم الآلة",
    "Company_Name": null,
    "Location": "نيويورك، الولايات المتحدة الأمريكية",
    "Experience_Level": "Senior",
    "Min_Years_of_Experience": 4,
    "Max_Years_of_Experience": null,
    "Salary": "١٥٠,٠٠٠ - ١٨٠,٠٠٠ دولار سنوياً",
    "Tech_Stack": [
        "Python",
        "GitHub Actions",
        "Jenkins",
        "MLflow",
        "Grafana",
        "Kubernetes",
        "Terraform",
        "AWS",
        "FastAPI"
    ],
    "Work_Model": "Hybrid",
    "Core_Responsibilities": [
        "بناء وصيانة خطوط أنابيب التكامل والنشر المستمر (CI/CD) لنماذج تعلم الآلة.",
        "مراقبة نماذج الإنتاج عن بُعد باستخدام ميزات MLflow وGrafana.",
        "إدارة النشر على عناقيد AWS وتنسيق النماذج مع فرق الواجهات الخلفية."
    ]
}
```


### Prompt Update: Fixing the Company Name & Tech Stack Bug

In [ ]:
job_extraction_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are an expert HR Data Parser and a bilingual (English-Arabic) AI Engineer.",
            "Your task is to extract specific information from an English Job Description based on a provided Pydantic schema.",
            "CRITICAL RULES:",
            "1. JSON Keys MUST remain strictly in English.",
            "2. JSON Values MUST be translated into professional Arabic.",
            "3. Technical skills, frameworks, and tools (e.g., Python, AWS, React) MUST be kept in English.",
            "4. ANTI-MISSING RULE (Tech Stack): Scan the ENTIRE document up to the very last word. Technical skills are often buried in the final requirements. You MUST NOT miss any tool, language, or framework.",
            "5. CONTEXTUAL DEDUCTION (Company Name): The Company Name might NOT have an explicit label like 'Company:'. You MUST carefully read the introductory sentences (e.g., 'At GlobalData Corp, we...') to deduce the company name before ever using null.",
            "6. Only use null if a specific piece of information (Salary, Years of Experience) is absolutely not mentioned anywhere in the text.",
            "7. Do not generate any introduction, conclusion, or conversational text. Output ONLY a valid JSON object."
        ])
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Job Description:",
            test_job_description.strip(),
            "",
            "## Pydantic Schema:",
            json.dumps(
                JobDetails.model_json_schema(), ensure_ascii=False
            ),
            "",
            "## Extracted JSON:",
            "```json\n{"
        ])
    }
]

In [ ]:
print(genereate_response(job_extraction_messages))

```json
{
    "Job_Title": "مهندس عمليات تعلم الآلة",
    "Company_Name": "GlobalData Corp",
    "Location": "نيويورك، الولايات المتحدة الأمريكية",
    "Experience_Level": "Senior",
    "Min_Years_of_Experience": 4,
    "Max_Years_of_Experience": null,
    "Salary": "١٥٠,٠٠٠ - ١٨٠,٠٠٠ دولار سنوياً",
    "Tech_Stack": [
        "Python",
        "GitHub Actions",
        "Jenkins",
        "MLflow",
        "Grafana",
        "Kubernetes",
        "Terraform",
        "AWS",
        "FastAPI"
    ],
    "Work_Model": "Hybrid",
    "Core_Responsibilities": [
        "بناء وصيانة خطوط أنابيب التكامل والنشر المستمر (CI/CD) لنماذج تعلم الآلة.",
        "مراقبة نماذج الإنتاج عن بُعد باستخدام ميزات MLflow وGrafana.",
        "إدارة النشر على عناقيد AWS وتنسيق النماذج مع فرق الواجهات الخلفية."
    ]
}
```


# Cost Estimation

In [9]:
system_message= "\n".join([
            "You are an expert HR Data Parser and a bilingual (English-Arabic) AI Engineer.",
            "Your task is to extract specific information from an English Job Description based on a provided Pydantic schema.",
            "CRITICAL RULES:",
            "1. JSON Keys MUST remain strictly in English.",
            "2. JSON Values MUST be translated into professional Arabic.",
            "3. Technical skills, frameworks, and tools (e.g., Python, AWS, React) MUST be kept in English.",
            "4. ANTI-MISSING RULE (Tech Stack): Scan the ENTIRE document up to the very last word. Technical skills are often buried in the final requirements. You MUST NOT miss any tool, language, or framework.",
            "5. CONTEXTUAL DEDUCTION (Company Name): The Company Name might NOT have an explicit label like 'Company:'. You MUST carefully read the introductory sentences (e.g., 'At GlobalData Corp, we...') to deduce the company name before ever using null.",
            "6. Only use null if a specific piece of information (Salary, Years of Experience) is absolutely not mentioned anywhere in the text.",
            "7. Do not generate any introduction, conclusion, or conversational text. Output ONLY a valid JSON object."
        ])

In [13]:
val_dataset_path = "/gdrive/MyDrive/llm-finetuning/Jobs_Dataset/val.json"

with open(val_dataset_path, "r", encoding="utf-8") as file:
    val_data = json.load(file)

all_jobs = [item["input"] for item in val_data if "input" in item]
num_samples = min(30, len(all_jobs))
test_batch = random.sample(all_jobs, num_samples)

input_tokens_count = 0
output_tokens_count = 0

start_time = time.time()

for job_desc in tqdm(test_batch):

    messages = [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": f"Extract the key information as JSON from:\n\n{job_desc}"
        }
    ]

    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_tokens_count += len(tokenizer.encode(prompt_text))

    response = genereate_response(messages)

    total_time = time.time() - start_time

    output_tokens_count += len(tokenizer.encode(response))

total_tokens = input_tokens_count + output_tokens_count

avg_time = total_time / num_samples
throughput = total_tokens / total_time

print("\n--- BENCHMARKING REPORT ---")
print(f"Total Requests Processed : {num_samples}")
print(f"Total Time Elapsed       : {total_time:.2f} seconds")
print(f"Average Time / Request   : {avg_time:.2f} seconds")
print(f"Throughput               : {throughput:.2f} tokens/second")
print("---------------------------")
print(f"Total Input Tokens       : {input_tokens_count:,}")
print(f"Total Output Tokens      : {output_tokens_count:,}")
print(f"Total Processed Tokens   : {total_tokens:,}")
print("---------------------------")

  0%|          | 0/30 [00:00<?, ?it/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- BENCHMARKING REPORT ---
Total Requests Processed : 30
Total Time Elapsed       : 594.27 seconds
Average Time / Request   : 19.81 seconds
Throughput               : 48.29 tokens/second
---------------------------
Total Input Tokens       : 20,656
Total Output Tokens      : 8,043
Total Processed Tokens   : 28,699
---------------------------


# vLLM

Online Serving

In [7]:
!nohup vllm serve "Qwen/Qwen2.5-1.5B-Instruct" \
    --enable-lora \
    --max-lora-rank 32 \
    --max-model-len 2048 \
    --trust-remote-code \
    --gpu-memory-utilization 0.85 \
    --lora-modules jobs_lora="abdoghazala7/Jobs" > vllm_server.log 2>&1 &

In [8]:
json_schema_str = json.dumps(JobDetails.model_json_schema())

url = "http://localhost:8000/v1/chat/completions"

headers = {
    "Content-Type": "application/json"
}

payload = {
    "model": "jobs_lora",
    "messages": [
        {
            "role": "system",
            "content": "You are an expert HR Data Parser. Extract information into the provided JSON schema.\nCRITICAL RULES:\n1. Translate values to Arabic, but keep Technical skills, frameworks, and tools strictly in English.\n2. ANTI-MISSING RULE: Scan the entire text up to the very last word so you don't miss any Tech Stack.\n3. CONTEXTUAL DEDUCTION: Read introductory sentences carefully to deduce the Company_Name before resorting to null."
        },
        {
            "role": "user",
            "content": f"Extract information from:\n\n{test_job_description}"
        }
    ],
    "temperature": 0.0,
    "max_tokens": 1024,
    "stop": ["<|im_end|>", "<|endoftext|>"],
    "guided_json": json_schema_str
}

response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    final_output = response.json()['choices'][0]['message']['content']
    print(final_output)

else:
    print(f"{response.status_code}")
    print(response.text)

```json
{
  "Company_Name": "GlobalData Corp",
  "Location": "نيويورك، الولايات المتحدة الأمريكية",
  "Experience_Level": "Senior",
  "Min_Years_of_Experience": 4,
  "Max_Years_of_Experience": null,
  "Salary": "من ١٥٠,٠٠٠ إلى ١٨٠,٠٠٠ دولار سنوياً",
  "Tech_Stack": [
    "GitHub Actions",
    "Jenkins",
    "MLflow",
    "Grafana",
    "Kubernetes",
    "Terraform",
    "AWS",
    "FastAPI",
    "Python",
    "PyTorch",
    "Docker",
    "Snowflake"
  ],
  "Work_Model": "Hybrid",
  "Core_Responsibilities": [
    "بناء وصيانة مسارات CI/CD المؤتمتة لنماذج تعلم الآلة.",
    "مراقبة نماذج الإنتاج في الوقت الفعلي باستخدام أدوات التدفق السلس.",
    "إدارة النماذج على عناقيد AWS وتوحيدها عبر Jenkins.",
    "التعاون مع فرق الواجهات الخلفية لنشر الخدمات المصغرة."
  ]
}
```


# Load Testing

In [9]:
%%writefile locust_test.py
import random
import json
from locust import HttpUser, task, between
from pydantic import BaseModel
from typing import List, Optional, Literal

ExperienceLevel = Literal["Junior", "Mid-level", "Senior", "Executive", "Not Specified"]
WorkModel = Literal["Remote", "On-site", "Hybrid", "Not Specified"]

class JobDetails(BaseModel):
    Job_Title: str
    Company_Name: Optional[str]
    Location: Optional[str]
    Experience_Level: ExperienceLevel
    Min_Years_of_Experience: Optional[int]
    Max_Years_of_Experience: Optional[int]
    Salary: Optional[str]
    Tech_Stack: List[str]
    Work_Model: WorkModel
    Core_Responsibilities: List[str]

json_schema_str = json.dumps(JobDetails.model_json_schema())

with open("/gdrive/MyDrive/llm-finetuning/Jobs_Dataset/val.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)
jobs_list = [item["input"] for item in val_data if "input" in item]

class VllmLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def extract_job(self):
        job_desc = random.choice(jobs_list)

        payload = {
            "model": "jobs_lora",
            "messages": [
                {
                    "role": "system",
                    "content": "You are an expert HR Data Parser. Extract information into the provided JSON schema.\nCRITICAL RULES:\n1. Translate values to Arabic, but keep Technical skills, frameworks, and tools strictly in English.\n2. ANTI-MISSING RULE: Scan the entire text up to the very last word so you don't miss any Tech Stack.\n3. CONTEXTUAL DEDUCTION: Read introductory sentences carefully to deduce the Company_Name before resorting to null."
                },
                {
                    "role": "user",
                    "content": f"Extract information from:\n\n{job_desc}"
                }
            ],
            "temperature": 0.0,
            "max_tokens": 1024,
            "guided_json": json_schema_str,
            "stop": ["<|im_end|>", "<|endoftext|>"]
        }

        with self.client.post("/v1/chat/completions", json=payload, catch_response=True) as response:
            if response.status_code == 200:
                usage = response.json().get("usage", {})

                with open("./vllm_load_stats.jsonl", "a", encoding="utf-8") as dest:
                    dest.write(json.dumps(usage) + "\n")

Writing locust_test.py


In [10]:
!locust --headless -f locust_test.py --host=http://localhost:8000 -u 20 -r 5 -t "60s" --html=locust_results.html

[2026-03-30 20:28:25,558] f05be66e30ce/INFO/locust.main: Starting Locust 2.43.3
[2026-03-30 20:28:25,559] f05be66e30ce/INFO/locust.main: Run time limit set to 60 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-03-30 20:28:25,560] f05be66e30ce/INFO/locust.runners: Ramping to 20 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /v1/chat/completions       1   1(100.00%) |     98      98      98     98 |    0.00        0.00
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
  

In [11]:
vllm_stats = []
with open("./vllm_load_stats.jsonl", "r") as file:
    for line in file:
        if line.strip():
             vllm_stats.append(json.loads(line.strip()))


total_requests = len(vllm_stats)
total_input_tokens = sum([rec.get('prompt_tokens', 0) for rec in vllm_stats])
total_output_tokens = sum([rec.get('completion_tokens', 0) for rec in vllm_stats])
total_tokens = total_input_tokens + total_output_tokens

total_time = 60.0
throughput = total_tokens / total_time
avg_time = total_time / total_requests if total_requests > 0 else 0

print("\n" + "="*50)
print(" VLLM LOAD TESTING REPORT (Concurrent Users: 20)")
print("="*50)
print(f"Total Requests Processed : {total_requests}")
print(f"Total Time Elapsed       : {total_time:.2f} seconds")
print(f"Average Time / Request   : {avg_time:.2f} seconds")
print(f"Throughput               : {throughput:.2f} tokens/second")
print("-" * 50)
print(f"Total Input Tokens       : {total_input_tokens:,}")
print(f"Total Output Tokens      : {total_output_tokens:,}")
print(f"Total Processed Tokens   : {total_tokens:,}")
print("="*50)


 VLLM LOAD TESTING REPORT (Concurrent Users: 20)
Total Requests Processed : 67
Total Time Elapsed       : 60.00 seconds
Average Time / Request   : 0.90 seconds
Throughput               : 772.65 tokens/second
--------------------------------------------------
Total Input Tokens       : 30,108
Total Output Tokens      : 16,251
Total Processed Tokens   : 46,359
